<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 9


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Order в C#, который будет представлять информацию о
заказах товаров или услуг. На основе этого класса разработать 2-3 производных
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из
классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;

try
{
    Item item1 = new Item("Помада", 450);
    Item item2 = new Item("Тушь", 200);
    Item item3 = new Item("Тональный крем", 1100);
    Item item4 = new Item("Румяна", 600);

    OnlineOrder onlineOrder = new OnlineOrder(111, "09.09.26", "nastya2405@mail.com");
    onlineOrder.AddItem(item1);
    onlineOrder.AddItem(item2);
    
    Console.WriteLine($"[Интернет-заказ]: ID: {onlineOrder.OrderId} | Дата: {onlineOrder.CreationDate} | Email: {onlineOrder.CustomerEmail}");
    onlineOrder.CalculateTotal();
    Console.WriteLine(new string('-', 75));

    PhysicalOrder physicalOrder = new PhysicalOrder(222, "09.09.26", "ул. Ленина, д. 10");
    physicalOrder.AddItem(item3);
    physicalOrder.AddItem(item4);

    Console.WriteLine($"[Курьерский заказ]: ID: {physicalOrder.OrderId} | Дата: {physicalOrder.CreationDate} | Адрес: {physicalOrder.DeliveryAddress}");
    physicalOrder.CalculateTotal();
    physicalOrder.RemoveItem(item3);
    physicalOrder.CalculateTotal();
    Console.WriteLine(new string('-', 75));

    SpecializedOrder specOrder = new SpecializedOrder(333, "09.09.26", "Карта лояльности", 15);
    Console.WriteLine($"[Спец-заказ]: ID: {specOrder.OrderId} | Условие: {specOrder.SpecialConditions} | Скидка: {specOrder.DiscountPercent}%");
    
    specOrder.MergeWithOrder(onlineOrder);
    specOrder.CalculateTotal();
    Console.WriteLine(new string('=', 75));

    Console.WriteLine("Попытка установить некорректную скидку...");
    specOrder.DiscountPercent = -5;
}
catch (Exception ex)
{
    Console.WriteLine($"Ошибка валидации данных -> {ex.Message}");
}

public class Item
{
    public string Name { get; set; }
    public int Amount { get; set; }

    public Item(string name, int amount)
    {
        Name = name;
        Amount = amount;
    }
}

public class Order
{
    private int _orderId;
    public string CreationDate { get; set; }
    public int TotalAmount { get; protected set; }
    protected List<Item> items = new List<Item>();

    public int OrderId
    {
        get { return _orderId; }
        set
        {
            if (value > 0) _orderId = value;
            else throw new ArgumentOutOfRangeException("ID заказа должен быть положительным числом!");
        }
    }

    public Order(int orderId, string creationDate)
    {
        OrderId = orderId;
        CreationDate = creationDate;
    }

    public virtual void AddItem(Item item)
    {
        items.Add(item);
    }

    public virtual void RemoveItem(Item item)
    {
        if (items.Contains(item))
        {
            items.Remove(item);
            Console.WriteLine($"Удален товар '{item.Name}' из заказа №{OrderId}");
        }
    }

    public virtual void CalculateTotal()
    {
        int sum = 0;
        foreach (var item in items)
        {
            sum += item.Amount;
        }
        TotalAmount = sum;
        Console.WriteLine($"Итоговая сумма заказа №{OrderId}: {TotalAmount} руб.");
    }
}

public class OnlineOrder : Order
{
    public string CustomerEmail { get; set; }

    public OnlineOrder(int orderId, string creationDate, string email) : base(orderId, creationDate)
    {
        CustomerEmail = email;
    }

    public override void AddItem(Item item)
    {
        base.AddItem(item);
        Console.WriteLine($"Товар '{item.Name}' добавлен. Код отправлен на {CustomerEmail}");
    }
}

public class PhysicalOrder : Order
{
    public string DeliveryAddress { get; set; }

    public PhysicalOrder(int orderId, string creationDate, string address) : base(orderId, creationDate)
    {
        DeliveryAddress = address;
    }

    public override void RemoveItem(Item item)
    {
        if (items.Contains(item))
        {
            base.RemoveItem(item);
            Console.WriteLine($"Возврат: Товар курьер заберет с адреса: {DeliveryAddress}");
        }
    }
}

public class SpecializedOrder : Order
{
    public string SpecialConditions { get; set; }
    private int _discountPercent;

    public int DiscountPercent
    {
        get { return _discountPercent; }
        set
        {
            if (value >= 0 && value <= 100) _discountPercent = value;
            else throw new ArgumentOutOfRangeException("Скидка должна быть в диапазоне от 0 до 100%!");
        }
    }

    public SpecializedOrder(int orderId, string creationDate, string conditions, int discount) : base(orderId, creationDate)
    {
        SpecialConditions = conditions;
        DiscountPercent = discount;
    }

    public void MergeWithOrder(Order otherOrder)
    {
        Console.WriteLine($"[Взаимодействие]: Заказ №{OrderId} объединяет товары из заказа №{otherOrder.OrderId}");
        foreach (var item in otherOrder.GetType().GetField("items", System.Reflection.BindingFlags.NonPublic | System.Reflection.BindingFlags.Instance)?.GetValue(otherOrder) as List<Item> ?? new List<Item>())
        {
            this.AddItem(item);
        }
    }

    public override void CalculateTotal()
    {
        int sum = 0;
        foreach (var item in items)
        {
            sum += item.Amount;
        }
        int discountSum = (sum * DiscountPercent) / 100;
        TotalAmount = sum - discountSum;
        Console.WriteLine($"Сумма со скидкой {DiscountPercent}% ({SpecialConditions}): {TotalAmount} руб.");
    }
}


Товар 'Помада' добавлен. Код отправлен на nastya2405@mail.com
Товар 'Тушь' добавлен. Код отправлен на nastya2405@mail.com
[Интернет-заказ]: ID: 111 | Дата: 09.09.26 | Email: nastya2405@mail.com
Итоговая сумма заказа №111: 650 руб.
---------------------------------------------------------------------------
[Курьерский заказ]: ID: 222 | Дата: 09.09.26 | Адрес: ул. Ленина, д. 10
Итоговая сумма заказа №222: 1700 руб.
Удален товар 'Тональный крем' из заказа №222
Возврат: Товар курьер заберет с адреса: ул. Ленина, д. 10
Итоговая сумма заказа №222: 600 руб.
---------------------------------------------------------------------------
[Спец-заказ]: ID: 333 | Условие: Карта лояльности | Скидка: 15%
[Взаимодействие]: Заказ №333 объединяет товары из заказа №111
Сумма со скидкой 15% (Карта лояльности): 553 руб.
Попытка установить некорректную скидку...
Ошибка валидации данных -> Specified argument was out of the range of valid values. (Parameter 'Скидка должна быть в диапазоне от 0 до 100%!')
